# 迁移学习与分阶段解冻

## 学习目标

能够加载预训练 ResNet、冻结主干、替换分类头，并使用分组学习率解冻最后一层。


## 概念模型与执行路径

预训练模型提供通用视觉特征。第一阶段只训练新分类头；第二阶段以更小学习率微调靠近输出的主干层，避免快速破坏已有表示。输入必须使用预训练权重对应的尺寸和归一化。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
from examples.transfer_learning import build_model, unfreeze_last_block
model = build_model(pretrained=False)  # 离线检查结构；真实课程默认下载预训练权重
trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable before unfreeze:", trainable_before)


### 实验 3


In [ ]:
unfreeze_last_block(model)
trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable after unfreeze:", trainable_after)
assert trainable_after > trainable_before


### 实验 4


In [ ]:
import torch
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
print([group["lr"] for group in optimizer.param_groups])


### 实验 5


In [ ]:
# 真实 CIFAR-10 训练：
# python 07-deep-learning/pytorch/examples/transfer_learning.py --epochs 4 --batch-size 32
# 离线结构冒烟需要已有 CIFAR 缓存；--quick 不下载预训练权重。


## 底层机制

冻结参数会阻止梯度存储，但冻结主干并不自动把 BatchNorm 切换为 eval。小数据微调时应谨慎处理 BatchNorm 统计量。参数组允许不同层使用不同学习率。


## 检查点

为什么分类头学习率通常高于已预训练的 layer4？解冻全部主干会带来什么风险？


## 试一试

比较只训练分类头和解冻 layer4 的验证准确率，同时记录训练时间和可训练参数量。


## 常见错误与调试

预训练模型使用错误归一化、一次性大幅更新所有层、替换分类头后仍把它冻结、用随机输入声称完成迁移学习。
